# Tool check: simulation

Runs each tool of this capability **directly, without the agent**, on the inputs you give. One section per tool: run the setup, then only the section you need.

In [ ]:
import os, sys, time, json
ROOT = os.path.abspath(os.path.join("..", ".."))      # notebooks/tools/ -> project root
sys.path.insert(0, ROOT)
from dynalyst.orchestration import tools as registry
from IPython.display import Image, display


def arguments(tool):
    """The tool's arguments, read from the tool itself."""
    schema = registry.schema_of(registry.all_tools()[tool])
    print(tool, "-- required:", schema.get("required", []))
    for name, spec in schema.get("properties", {}).items():
        print(f"  {name} ({spec.get('type', 'any')}): {(spec.get('description') or '')[:150]}")


def run_tool(tool, args, pasted=""):
    """Run one tool alone: your ARGS, or the agent's call pasted from its log."""
    if pasted.strip():
        args = json.loads(pasted)
    print(tool, args)
    t = time.time()
    raw = registry.execute(tool, args)
    return show(tool, raw, time.time() - t)


def show(tool, raw, seconds):
    """The result: error first, then each field (long ones shortened), then plots."""
    print(f"{tool} ran in {seconds:.1f} s")
    try:
        data = json.loads(raw)
    except (TypeError, ValueError):
        print(str(raw)[:3000])
        return None
    if isinstance(data, dict):
        if data.get("error"):
            print("ERROR:", data["error"])
        for key, value in data.items():
            text = json.dumps(value, default=str)
            print(f"- {key}: {text[:400]}{' ...' if len(text) > 400 else ''}")
        for key, value in data.items():
            if isinstance(value, str) and value.lower().endswith(".png"):
                path = value if os.path.isabs(value) else os.path.join(ROOT, value)
                if os.path.exists(path):
                    print(key)
                    display(Image(filename=path))
    else:
        print(json.dumps(data, default=str)[:3000])
    return data

TOOLS = ['simulate_model', 'compare_initial_conditions', 'plot_phase_plane', 'custom_plot']

## `simulate_model`


In [ ]:
arguments("simulate_model")
# Fill in what you need (required: ['model_name_or_path']), OR paste the agent's call from its log
# (the {...} after "SELECT simulate_model(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string  REQUIRED: Model name (a file in the model folders, with or without .xml) or full path to an .xml SBML file.
    # "t_end": ...,   # number: End time for simulation, in the model's own time unit (inspect_model's time_unit; convert the user's
    # "n_points": ...,   # integer: Number of time points. Default 200.
    # "species": ...,   # array: Optional list of species names to restrict the PLOT to (e.g. ['S1']). Returned data still includes a
    # "initial_conditions": ...,   # object: Optional dict of {species_name: value} overriding the model's default starting concentrations for ju
    # "parameters": ...,   # object: Optional dict of {parameter_name: value} overriding the model's parameter values for this run only, 
}
PASTED = ""
result = run_tool("simulate_model", ARGS, PASTED)

## `compare_initial_conditions`


In [ ]:
arguments("compare_initial_conditions")
# Fill in what you need (required: ['model_name_or_path', 'initial_conditions_list']), OR paste the agent's call from its log
# (the {...} after "SELECT compare_initial_conditions(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string  REQUIRED: Model name or path to an SBML file.
    # "initial_conditions_list": ...,   # array  REQUIRED: List of dicts, each specifying only the species being changed for that run, e.g. [{'S1': 500, 'S2': 
    # "t_end": ...,   # number: End time for simulation, in the model's own time unit (inspect_model's time_unit; convert the user's
    # "n_points": ...,   # integer: Number of time points. Default 200.
    # "species": ...,   # array: Optional list of species names to plot (e.g. ['S1']). Omit to plot all species for every condition.
}
PASTED = ""
result = run_tool("compare_initial_conditions", ARGS, PASTED)

## `plot_phase_plane`


In [ ]:
arguments("plot_phase_plane")
# Fill in what you need (required: ['model_name_or_path', 'x_species', 'y_species']), OR paste the agent's call from its log
# (the {...} after "SELECT plot_phase_plane(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string  REQUIRED: Any model name or SBML path.
    # "x_species": ...,   # string  REQUIRED: Species for the x-axis.
    # "y_species": ...,   # string  REQUIRED: Species for the y-axis.
    # "t_end": ...,   # number: End time for simulation, in the model's own time unit (inspect_model's time_unit; convert the user's
    # "n_points": ...,   # integer: Number of time points. Default 500.
    # "initial_conditions": ...,   # object: Optional {species: value} overrides.
    # "parameter_overrides": ...,   # object: Optional {parameter: value} overrides.
}
PASTED = ""
result = run_tool("plot_phase_plane", ARGS, PASTED)

## `custom_plot`


In [ ]:
arguments("custom_plot")
# Fill in what you need (required: ['model_name_or_path']), OR paste the agent's call from its log
# (the {...} after "SELECT custom_plot(") into PASTED.
ARGS = {
    # "model_name_or_path": ...,   # string  REQUIRED: Model name or SBML path.
    # "species": ...,   # array: Species to plot (y-axis), by their names in the model; omit to plot every species.
    # "x_axis": ...,   # string: 'time' (default) or a species name for a phase-style plot.
    # "t_end": ...,   # number: Simulation end time, in the model's own time unit (inspect_model's time_unit; convert the user's hou
    # "n_points": ...,   # integer: Time points. Default 500.
    # "t_start": ...,   # number: Start of the plotted time window, in the model's own time unit. Default 0.
    # "initial_conditions": ...,   # object: {species: value} overrides (single condition).
    # "parameter_overrides": ...,   # object: {parameter: value} overrides (single condition).
    # "conditions": ...,   # array: Overlay several runs: [{'label': str, 'initial_conditions': {...}, 'parameter_overrides': {...}}, ..
    # "log_x": ...,   # boolean: 
    # "log_y": ...,   # boolean: 
    # "title": ...,   # string: 
}
PASTED = ""
result = run_tool("custom_plot", ARGS, PASTED)

## The agent on the same request (optional)

In [ ]:
# Optional: ask the agent the same thing and compare (needs the API key).
QUERY = ""                       # e.g. the request that went wrong
if QUERY.strip():
    from dynalyst.agents.unified_agent import run
    t = time.time()
    rec = run(QUERY)
    print(f"agent: {time.time() - t:.1f} s\n")
    calls = [s for s in rec.steps if s.get("tool") in TOOLS]
    print(f"the agent made {len(calls)} call(s) to these tools:")
    for a in calls:
        print(" ", a.get("tool"), json.dumps(a.get("arguments"), default=str)[:400],
              "|", a.get("status"), (a.get("error") or "")[:200])
    print("\nanswer:\n", rec.final_answer[:2000])

**How to read it**
- The tool fails or gives a wrong result here, with the agent's arguments: the **tool** is the cause.
- The tool works here, but the chat went wrong: the **agent** is the cause (wrong arguments,
  wrong tool, or a wrong reading of the result). Compare with the agent's calls above.
- The tool works here only with different arguments: the agent passed wrong arguments.